# GMN v2 — resumable runner for free Colab GPUs

**Every session (including after a disconnect): run cells 1 → 2 → 3. Nothing else.**

- Nothing that is finished is ever recomputed: the parsed dataset is cached on Drive, finished runs are skipped, an interrupted run continues from its last 5-epoch checkpoint, and one-off steps leave a marker.
- Cell 3 first prints what is done and what is left, then continues.
- Everything is written to Google Drive (`gmn_results_v2_dedup`, `..._asreleased`, `..._lmdhg`), and a log to `gmn_results_v2_dedup/pipeline.log`.
- Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# CELL 1 — mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 2 — install the code (always from the zip on Drive, so you never run an old copy)
import os, shutil, subprocess, sys
CODE_ZIP = '/content/drive/MyDrive/GMN_v2_code_and_manuscript.zip'
shutil.rmtree('/content/gmn-v2', ignore_errors=True); shutil.rmtree('/content/gmn', ignore_errors=True)
subprocess.run(['unzip', '-q', '-o', CODE_ZIP, '-d', '/content'], check=True)
%cd /content/gmn-v2
!pip -q install -r requirements.txt
for _m in [m for m in sys.modules if m == 'gmn' or m.startswith('gmn.')]: del sys.modules[_m]
sys.path.insert(0, '/content/gmn-v2')
import gmn; print('code version', gmn.__version__)
assert gmn.__version__ >= '2.4.0', 'old zip on Drive -- upload the latest GMN_v2_code_and_manuscript.zip'
!nvidia-smi -L

In [ ]:
# CELL 3 — show progress, then run / continue everything (safe to re-run any time)
# ── your paths (edit here only) ──────────────────────────────────────────
LEAP_ZIP  = '/content/drive/MyDrive/GMN_Datasets.zip'      # zip that contains .../LeapGestureDB
LMDHG_ZIP = '/content/drive/MyDrive/LMDHG.zip'
SWEEP_CSV = '/content/drive/MyDrive/gmn_results/sweep.csv'
RES       = '/content/drive/MyDrive/gmn_results_v2_dedup'
import os, glob, subprocess
CACHE = f'{RES}/_cache/leapgesturedb_T100_n6600.npz'

def count_txt(d):
    return sum(f.lower().endswith('.txt') for _, _, fs in os.walk(d) for f in fs)

def find_leap(root):
    """Folder named LeapGestureDB (any depth) with the most .txt files."""
    cands = [dp for dp, dn, _ in os.walk(root) if os.path.basename(dp).lower() == 'leapgesturedb']
    return max(cands, key=count_txt) if cands else None

if os.path.exists(CACHE):
    LEAP_DIR = '/content/unused'          # dataset already parsed and cached on Drive
    print('validated dataset cache on Drive -> no unzip needed')
else:
    UNZ = '/content/leap_unzipped'
    if not find_leap(UNZ):
        print('unzipping', LEAP_ZIP, '(once per session until the cache exists) ...')
        os.makedirs(UNZ, exist_ok=True)
        subprocess.run(['unzip', '-q', '-o', LEAP_ZIP, '-d', UNZ], check=True)
        # LeapGestureDB may itself be a zip inside the zip
        for z in glob.glob(UNZ + '/**/*.zip', recursive=True):
            if 'leap' in os.path.basename(z).lower():
                subprocess.run(['unzip', '-q', '-o', z, '-d', os.path.dirname(z)], check=True)
    LEAP_DIR = find_leap(UNZ)
    assert LEAP_DIR, f'no folder named LeapGestureDB inside {LEAP_ZIP}; folders found: ' + str(
        sorted({os.path.relpath(dp, UNZ) for dp, _, _ in os.walk(UNZ)})[:20])
    print('LeapGestureDB found at', LEAP_DIR, 'with', count_txt(LEAP_DIR), '.txt files (need 6600)')

for p in (LMDHG_ZIP, SWEEP_CSV):
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + p)
assert os.path.exists(SWEEP_CSV), 'fix SWEEP_CSV'
ARGS = f'--leap_drive "{LEAP_DIR}" --lmdhg_zip "{LMDHG_ZIP}" --sweep_csv "{SWEEP_CSV}" --res "{RES}"'
%cd /content/gmn-v2
!python scripts/pipeline.py $ARGS --status
!python scripts/pipeline.py $ARGS

### Optional
- Only one part: `!python scripts/pipeline.py --stages main` (stages: data audit recipe main ablations analysis asreleased lmdhg paper).
- Fewer runs if GPU time is short: `--seeds 0 1` for the main table.
- Check a single LMDHG file:
```python
from gmn.data_lmdhg import inspect_mat, load_lmdhg_segments
D = load_lmdhg_segments('/content/LMDHG')   # after the lmdhg stage has unzipped it
```
- Paths are the defaults in `scripts/pipeline.py` (`--leap_drive`, `--lmdhg_zip`, `--sweep_csv`, `--res`); pass them explicitly if yours differ.